Q. Bayesian Estimation of a User Ability Parameter from Item Responses

In [11]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from IPython.display import display, HTML

np.random.seed(42)

display(HTML("""
<h1>Bayesian Sequential Updating using the 2PL Item Response Model</h1>

<h2>Task 1</h2>

<p>The probability of answering an item correctly is:</p>

<p>
$$
P(Y_i=1|\\Theta=\\theta)
=
p_i(\\theta)
=
\\frac{1}{1+\\exp[-a_i(\\theta-b_i)]}
$$
</p>

<p>
where:
</p>

<ul>
<li><b>a_i</b> = discrimination parameter</li>
<li><b>b_i</b> = difficulty parameter</li>
</ul>

<p>
Increasing the discrimination parameter makes the logistic curve steeper.
</p>

<p>
Changing the difficulty parameter shifts the curve horizontally.
</p>

<ul>
<li>Larger b shifts the curve to the right.</li>
<li>Smaller b shifts the curve to the left.</li>
</ul>

<hr>

<h2>Task 2</h2>

<p>For one observation:</p>

<p>
$$
L(y_k|\\theta)
=
p_k(\\theta)^{y_k}
(1-p_k(\\theta))^{1-y_k}
$$
</p>

<p>
Since responses are conditionally independent:
</p>

<p>
$$
L(\\mathbf{y}^{(k)}|\\theta)
=
\\prod_{i=1}^{k}
p_i(\\theta)^{y_i}
(1-p_i(\\theta))^{1-y_i}
$$
</p>

<hr>

<h2>Task 3</h2>

<p>Bayes theorem gives:</p>

<p>
$$
f(\\theta|\\mathbf{y}^{(k)})
\\propto
L(y_k|\\theta)
f(\\theta|\\mathbf{y}^{(k-1)})
$$
</p>

<p>
or equivalently:
</p>

<p>
$$
f(\\theta|\\mathbf{y}^{(k)})
\\propto
p_k(\\theta)^{y_k}
(1-p_k(\\theta))^{1-y_k}
f(\\theta|\\mathbf{y}^{(k-1)})
$$
</p>

<p>
The posterior after step k-1 becomes the prior for step k.
</p>

"""))


def logistic(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))


theta = np.linspace(-4, 4, 500)


fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Different Discrimination Parameters",
        "Effect of Difficulty Parameter"
    )
)


fig.add_trace(
    go.Scatter(
        x=theta,
        y=logistic(theta, 0.5, 0),
        mode="lines",
        name="a=0.5, b=0"
    ),
    row=1,
    col=1
)


fig.add_trace(
    go.Scatter(
        x=theta,
        y=logistic(theta, 2.0, 0),
        mode="lines",
        name="a=2.0, b=0"
    ),
    row=1,
    col=1
)


for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=logistic(theta,1.2,b),
            mode="lines",
            name=f"a=1.2, b={b}"
        ),
        row=1,
        col=2
    )


fig.update_xaxes(title="Ability θ")
fig.update_yaxes(title="Probability of Correct Response")

fig.update_layout(
    height=550,
    width=1100,
    template="plotly_white",
    title="2PL Item Response Curves"
)

fig.show()


print("="*70)
print("INTERPRETATION")
print("="*70)

print("""
Increasing the discrimination parameter (a)
makes the curve steeper, meaning the item
better separates users with different abilities.

Changing the difficulty parameter (b)
does not change the curve shape.

Instead:

b = -1 shifts the curve left
b = 0 keeps the curve centered
b = +1 shifts the curve right

A larger difficulty requires a higher ability
before the probability of a correct response
becomes high.
""")


theta_grid = np.linspace(-4,4,801)


prior = norm.pdf(theta_grid,0,1)
prior /= np.trapezoid(prior,theta_grid)


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior,
        mode="lines",
        name="Prior"
    )
)


fig.update_layout(
    template="plotly_white",
    title="Initial Prior Distribution N(0,1)",
    xaxis_title="Ability θ",
    yaxis_title="Density"
)


fig.show()


print("="*70)
print("Initial Prior Successfully Constructed")
print("="*70)


from IPython.display import display, HTML

display(HTML("""
<h1>Task 4</h1>

<p>Suppose the current response is correct (y<sub>k</sub> = 1).</p>

<p>The likelihood becomes:</p>

<p>
$$
L(y_k=1|\\theta)=p_k(\\theta)
$$
</p>

<p>Hence:</p>

<p>
$$
f(\\theta|\\mathbf y^{(k)})
\\propto
p_k(\\theta)
f(\\theta|\\mathbf y^{(k-1)})
$$
</p>

<p>
For a difficult item (b<sub>k</sub> large), the probability
</p>

<p>
$$
p_k(\\theta)
=
\\frac{1}{1+\\exp[-a_k(\\theta-b_k)]}
$$
</p>

<p>
is large only for larger values of θ.
</p>

<p>
Therefore, multiplying the previous posterior by this likelihood
shifts the posterior density toward higher ability values.
</p>

<p>
The posterior peak moves to the right, indicating increased belief
that the learner possesses higher latent ability.
</p>

<hr>

<h1>Task 5</h1>

<p>
The discrimination parameter controls how informative an item is.
</p>

<h3>Large discrimination:</h3>

<ul>
<li>Very steep likelihood</li>
<li>Posterior becomes narrower</li>
<li>Posterior variance decreases</li>
<li>Confidence increases</li>
</ul>

<h3>Small discrimination:</h3>

<ul>
<li>Flatter likelihood</li>
<li>Posterior changes only slightly</li>
<li>Posterior remains wide</li>
<li>Uncertainty stays relatively high</li>
</ul>

<hr>

<h1>Task 6</h1>

<h2>Grid Approximation Algorithm</h2>

<ol>
<li>Construct a fixed grid of ability values.</li>
<li>Initialize the prior density.</li>
<li>After each response evaluate the likelihood over the grid.</li>
<li>Multiply the previous posterior by the likelihood.</li>
<li>Normalize using numerical integration.</li>
<li>Compute posterior mean and MAP.</li>
<li>Store the posterior for the next update.</li>
</ol>

"""))


def likelihood(theta, y, a, b):
    p = logistic(theta, a, b)
    return (p ** y) * ((1 - p) ** (1 - y))


posterior = prior.copy()

theta_example = 1.2
a_example = 1.8
b_example = 0.8


like_correct = likelihood(theta_grid, 1, a_example, b_example)

posterior_correct = posterior * like_correct
posterior_correct /= np.trapezoid(posterior_correct, theta_grid)


fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=posterior,
        name="Previous Posterior",
        mode="lines"
    )
)


fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=posterior_correct,
        name="Updated Posterior",
        mode="lines"
    )
)


fig.add_vline(
    x=b_example,
    line_dash="dash",
    annotation_text="Difficulty"
)


fig.update_layout(
    template="plotly_white",
    title="Posterior Shift After Correct Response",
    xaxis_title="Ability θ",
    yaxis_title="Density"
)


fig.show()


print("="*70)
print("Dynamic Shift Interpretation")
print("="*70)

print("""
A correct answer on a difficult item assigns much larger
likelihood values to higher ability levels.

Multiplying the previous posterior by this likelihood
moves the posterior peak toward larger θ values.

This indicates that the learner is believed to possess
greater latent ability than before observing the item.
""")


a_values = [0.5,1.0,2.5]

fig = go.Figure()


for a in a_values:

    like = likelihood(theta_grid,1,a,0)

    post = posterior*like
    post /= np.trapezoid(post,theta_grid)

    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=post,
            mode="lines",
            name=f"a={a}"
        )
    )


fig.update_layout(
    template="plotly_white",
    title="Effect of Discrimination on Posterior Sharpness",
    xaxis_title="Ability θ",
    yaxis_title="Density"
)


fig.show()


print("="*70)
print("Sharpness Interpretation")
print("="*70)

print("""
As the discrimination parameter increases,
the posterior distribution becomes noticeably
narrower and taller.

This corresponds to a reduction in posterior
variance and greater confidence in the estimated
ability.

Low discrimination items contribute relatively
little information, producing only minor changes
to the posterior distribution.
""")


def bayesian_update(previous_posterior,y,a,b):

    like = likelihood(theta_grid,y,a,b)

    new_posterior = previous_posterior*like

    area = np.trapezoid(new_posterior,theta_grid)

    new_posterior /= area

    return new_posterior


print("="*70)
print("Sequential Bayesian Grid Update Algorithm")
print("="*70)


print("""
Step 1 : Evaluate likelihood on θ grid

Step 2 : Multiply previous posterior by likelihood

Step 3 : Compute area using trapezoidal integration

Step 4 : Divide every density value by this area

Step 5 : Store normalized posterior

Step 6 : Repeat after every new response
""")


example = bayesian_update(prior,1,1.5,0.5)


fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior,
        name="Prior"
    )
)


fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=example,
        name="Normalized Posterior"
    )
)


fig.update_layout(
    template="plotly_white",
    title="Numerical Bayesian Grid Update",
    xaxis_title="Ability θ",
    yaxis_title="Density"
)


fig.show()
from IPython.display import display, Markdown

display(Markdown(r"""
# Task 7

The user's true latent ability is assumed to be:

$$
\theta_{\mathrm{true}} = 0.75
$$

Twenty items are generated sequentially.

For each item:

$$
a_k \sim U(0.5,2.0)
$$

$$
b_k \sim N(0,1)
$$

The response is simulated from:

$$
Y_k \sim Bernoulli(p_k(\theta_{\mathrm{true}}))
$$

After every response, the posterior distribution is updated using Bayes' rule.

The following estimators are tracked:

---

## Posterior Mean

$$
\hat{\theta}_{Bayes}
=
E[\Theta|\mathbf{y}]
$$

---

## Maximum A Posteriori (MAP)

$$
\hat{\theta}_{MAP}
=
\arg\max_{\theta}
f(\theta|\mathbf{y})
$$

---
"""))

INTERPRETATION

Increasing the discrimination parameter (a)
makes the curve steeper, meaning the item
better separates users with different abilities.

Changing the difficulty parameter (b)
does not change the curve shape.

Instead:

b = -1 shifts the curve left
b = 0 keeps the curve centered
b = +1 shifts the curve right

A larger difficulty requires a higher ability
before the probability of a correct response
becomes high.



Initial Prior Successfully Constructed


Dynamic Shift Interpretation

A correct answer on a difficult item assigns much larger
likelihood values to higher ability levels.

Multiplying the previous posterior by this likelihood
moves the posterior peak toward larger θ values.

This indicates that the learner is believed to possess
greater latent ability than before observing the item.



Sharpness Interpretation

As the discrimination parameter increases,
the posterior distribution becomes noticeably
narrower and taller.

This corresponds to a reduction in posterior
variance and greater confidence in the estimated
ability.

Low discrimination items contribute relatively
little information, producing only minor changes
to the posterior distribution.

Sequential Bayesian Grid Update Algorithm

Step 1 : Evaluate likelihood on θ grid

Step 2 : Multiply previous posterior by likelihood

Step 3 : Compute area using trapezoidal integration

Step 4 : Divide every density value by this area

Step 5 : Store normalized posterior

Step 6 : Repeat after every new response




# Task 7

The user's true latent ability is assumed to be:

$$
\theta_{\mathrm{true}} = 0.75
$$

Twenty items are generated sequentially.

For each item:

$$
a_k \sim U(0.5,2.0)
$$

$$
b_k \sim N(0,1)
$$

The response is simulated from:

$$
Y_k \sim Bernoulli(p_k(\theta_{\mathrm{true}}))
$$

After every response, the posterior distribution is updated using Bayes' rule.

The following estimators are tracked:

---

## Posterior Mean

$$
\hat{\theta}_{Bayes}
=
E[\Theta|\mathbf{y}]
$$

---

## Maximum A Posteriori (MAP)

$$
\hat{\theta}_{MAP}
=
\arg\max_{\theta}
f(\theta|\mathbf{y})
$$

---


Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

In [12]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
from IPython.display import display, Markdown

np.random.seed(42)


display(Markdown(r"""

# Bayesian Sequential Updating of Advertisement Click-Through Rate (CTR)

## Task 1: Structural Probability and Properties

The probability density function of a Beta distribution is:

$$
f(\theta|\alpha,\beta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1}
$$

where:

- $\alpha$ controls the accumulated evidence for clicks.
- $\beta$ controls the accumulated evidence for non-clicks.

The Beta distribution is plotted for:

- Uniform distribution: $(\alpha=1,\beta=1)$
- Right-skewed distribution: $(\alpha=2,\beta=8)$
- Left-skewed distribution: $(\alpha=8,\beta=2)$

Changing the balance between $\alpha$ and $\beta$ shifts the center of mass.

A larger $\alpha$ increases probability density near 1, while a larger
$\beta$ increases density near 0.

---

"""))


theta = np.linspace(0,1,500)


fig = go.Figure()


parameters = [
    (1,1,"α=1, β=1"),
    (2,8,"α=2, β=8"),
    (8,2,"α=8, β=2")
]


for a,b,label in parameters:

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode="lines",
            name=label
        )
    )


fig.update_layout(
    template="plotly_white",
    title="Beta Distribution Shapes",
    xaxis_title="CTR θ",
    yaxis_title="Density"
)


fig.show()



display(Markdown(r"""

# Task 2: Sequential Likelihood and Joint History

For a single observation:

$$
L(y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}
$$


For the complete observation history:

$$
L(\mathbf{y}^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}
$$


---

# Task 3: Beta-Binomial Conjugacy

The recursive Bayesian update is:

$$
f(\theta|\mathbf{y}^{(k)})
\propto
L(y_k|\theta)
f(\theta|\mathbf{y}^{(k-1)})
$$


Assuming:

$$
\Theta|\mathbf{y}^{(k-1)}
\sim
Beta(\alpha_{k-1},\beta_{k-1})
$$


The updated parameters become:

$$
\alpha_k
=
\alpha_{k-1}+y_k
$$


$$
\beta_k
=
\beta_{k-1}+(1-y_k)
$$


Therefore:

$$
\Theta|\mathbf{y}^{(k)}
\sim
Beta(\alpha_k,\beta_k)
$$


The posterior mean is:

$$
E[\Theta|\mathbf{y}^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
$$


"""))



display(Markdown(r"""

# Task 4: Dynamic Shifting Mechanics

For a click:

$$
y_k=1
$$

the update becomes:

$$
\alpha_k=\alpha_{k-1}+1
$$

which increases the density toward higher values of $\theta$.


For a non-click:

$$
y_k=0
$$

the update becomes:

$$
\beta_k=\beta_{k-1}+1
$$

which shifts the density toward lower values of $\theta$.


Unlike non-conjugate models such as the 2PL IRT model,
the Beta-Binomial model has a closed-form solution and does
not require numerical grid approximation.

---

# Task 5: Running Point Estimators

Posterior Mean:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
$$


MAP estimate:

$$
\hat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
$$

for:

$$
\alpha_k>1,\beta_k>1
$$


"""))



theta_true = 0.35

n = 100

alpha = 1
beta_parameter = 1


bayes_estimates = [alpha/(alpha+beta_parameter)]

map_estimates = [0.5]

responses = []


for k in range(n):

    random_value = np.random.uniform(0,1)

    if random_value < theta_true:
        y = 1
    else:
        y = 0


    responses.append(y)


    alpha = alpha + y

    beta_parameter = beta_parameter + (1-y)


    bayes = alpha/(alpha+beta_parameter)


    if alpha > 1 and beta_parameter > 1:
        map_value = (alpha-1)/(alpha+beta_parameter-2)
    else:
        map_value = bayes


    bayes_estimates.append(bayes)

    map_estimates.append(map_value)



steps = np.arange(0,n+1)



fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Posterior Mean"
    )
)


fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)


fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35"
)


fig.update_layout(
    template="plotly_white",
    title="Sequential Bayesian CTR Estimation",
    xaxis_title="Number of Impressions",
    yaxis_title="Estimated CTR",
    height=600
)


fig.show()



print("="*80)
print("SIMULATION RESULTS")
print("="*80)

print(f"True CTR                 : {theta_true:.3f}")
print(f"Final Posterior Mean     : {bayes_estimates[-1]:.3f}")
print(f"Final MAP Estimate       : {map_estimates[-1]:.3f}")

print()

print("="*80)
print("CONVERGENCE ANALYSIS")
print("="*80)


print("""
Initially, the estimates are strongly influenced by the
uniform prior Beta(1,1), which represents complete uncertainty.

As more impressions are observed, each click increases alpha
and each non-click increases beta.

The accumulated evidence gradually dominates the initial prior.

Both the Posterior Mean and MAP estimates converge toward the
true CTR value of 0.35 as the number of impressions increases.

The decreasing difference between the estimates and the true
CTR demonstrates that Bayesian sequential learning improves
parameter estimation with additional observations.

Therefore, continuous evidence accumulation allows the platform
to adaptively learn the advertisement performance over time.
""")



# Bayesian Sequential Updating of Advertisement Click-Through Rate (CTR)

## Task 1: Structural Probability and Properties

The probability density function of a Beta distribution is:

$$
f(\theta|\alpha,\beta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1}
$$

where:

- $\alpha$ controls the accumulated evidence for clicks.
- $\beta$ controls the accumulated evidence for non-clicks.

The Beta distribution is plotted for:

- Uniform distribution: $(\alpha=1,\beta=1)$
- Right-skewed distribution: $(\alpha=2,\beta=8)$
- Left-skewed distribution: $(\alpha=8,\beta=2)$

Changing the balance between $\alpha$ and $\beta$ shifts the center of mass.

A larger $\alpha$ increases probability density near 1, while a larger
$\beta$ increases density near 0.

---





# Task 2: Sequential Likelihood and Joint History

For a single observation:

$$
L(y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}
$$


For the complete observation history:

$$
L(\mathbf{y}^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}
$$


---

# Task 3: Beta-Binomial Conjugacy

The recursive Bayesian update is:

$$
f(\theta|\mathbf{y}^{(k)})
\propto
L(y_k|\theta)
f(\theta|\mathbf{y}^{(k-1)})
$$


Assuming:

$$
\Theta|\mathbf{y}^{(k-1)}
\sim
Beta(\alpha_{k-1},\beta_{k-1})
$$


The updated parameters become:

$$
\alpha_k
=
\alpha_{k-1}+y_k
$$


$$
\beta_k
=
\beta_{k-1}+(1-y_k)
$$


Therefore:

$$
\Theta|\mathbf{y}^{(k)}
\sim
Beta(\alpha_k,\beta_k)
$$


The posterior mean is:

$$
E[\Theta|\mathbf{y}^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
$$






# Task 4: Dynamic Shifting Mechanics

For a click:

$$
y_k=1
$$

the update becomes:

$$
\alpha_k=\alpha_{k-1}+1
$$

which increases the density toward higher values of $\theta$.


For a non-click:

$$
y_k=0
$$

the update becomes:

$$
\beta_k=\beta_{k-1}+1
$$

which shifts the density toward lower values of $\theta$.


Unlike non-conjugate models such as the 2PL IRT model,
the Beta-Binomial model has a closed-form solution and does
not require numerical grid approximation.

---

# Task 5: Running Point Estimators

Posterior Mean:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
$$


MAP estimate:

$$
\hat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
$$

for:

$$
\alpha_k>1,\beta_k>1
$$




SIMULATION RESULTS
True CTR                 : 0.350
Final Posterior Mean     : 0.412
Final MAP Estimate       : 0.410

CONVERGENCE ANALYSIS

Initially, the estimates are strongly influenced by the
uniform prior Beta(1,1), which represents complete uncertainty.

As more impressions are observed, each click increases alpha
and each non-click increases beta.

The accumulated evidence gradually dominates the initial prior.

Both the Posterior Mean and MAP estimates converge toward the
true CTR value of 0.35 as the number of impressions increases.

The decreasing difference between the estimates and the true
CTR demonstrates that Bayesian sequential learning improves
parameter estimation with additional observations.

Therefore, continuous evidence accumulation allows the platform
to adaptively learn the advertisement performance over time.



# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In [13]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
from IPython.display import display, Markdown

np.random.seed(42)


display(Markdown(r"""

# Bayesian Sequential Updating for Structural Health Monitoring (SHM)

## Task 1: Prior Belief Boundaries

The initial belief about the remaining stiffness efficiency is:

$$
\Theta \sim Beta(8,1.5)
$$


The probability density function is:

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1}
$$


The expected prior stiffness efficiency is:

$$
E[\Theta^{(0)}]
=
\frac{\alpha}{\alpha+\beta}
$$


For:

$$
\alpha=8,\beta=1.5
$$


$$
E[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
0.842
$$


This prior represents a healthy structure because the density is concentrated
near θ = 1, indicating that most manufactured components are expected to
operate close to their nominal stiffness.

"""))



theta_grid = np.linspace(0.01,1,1000)


alpha_prior = 8
beta_prior = 1.5


prior = beta.pdf(theta_grid,alpha_prior,beta_prior)

prior /= np.trapezoid(prior,theta_grid)



fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior,
        mode="lines",
        name="Beta(8,1.5) Prior"
    )
)


fig.update_layout(
    template="plotly_white",
    title="Initial Structural Health Prior Distribution",
    xaxis_title="Remaining Stiffness Efficiency θ",
    yaxis_title="Density"
)


fig.show()



display(Markdown(r"""

# Task 2: Structural Likelihood Formulation


The sensor model is:

$$
y_k=\theta K_{nominal}e^{\epsilon_k}
$$


where:


$$
\epsilon_k\sim N(0,\sigma^2)
$$


Taking logarithms:


$$
ln(y_k)
=
ln(\theta K_{nominal})
+
\epsilon_k
$$


Therefore:


$$
ln(y_k)
\sim
N(ln(\theta K_{nominal}),\sigma^2)
$$


The likelihood of a single measurement is:


$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
exp
\left(
-\frac{
(ln(y_k)-ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right)
$$


The joint likelihood is:


$$
L(\mathbf y^{(k)}|\theta)
=
\prod_{i=1}^{k}
L(y_i|\theta)
$$


"""))



display(Markdown(r"""

# Task 3: Non-Conjugate Bayesian Grid Update


The posterior recursion is:


$$
f(\theta|\mathbf y^{(k)})
\propto
L(y_k|\theta)
f(\theta|\mathbf y^{(k-1)})
$$


Unlike Beta-Binomial models, the Beta prior combined with a
log-normal likelihood does not produce another Beta distribution.

Therefore, a closed-form analytical posterior does not exist.


Numerical grid approximation is required to represent the posterior
distribution.

"""))



display(Markdown(r"""

# Task 4: Running Point Estimates


The posterior mean is obtained using numerical integration:


$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{
\int_0^1
\theta f(\theta|\mathbf y^{(k)})d\theta
}
{
\int_0^1
f(\theta|\mathbf y^{(k)})d\theta
}
$$


The MAP estimate is:


$$
\hat{\theta}_{MAP}^{(k)}
=
\arg\max_{\theta}
f(\theta|\mathbf y^{(k)})
$$


"""))



display(Markdown(r"""

# Task 5: Grid Approximation Algorithm


1. Construct a bounded grid:

$$
\theta \in [0.01,1]
$$


2. Initialize the Beta prior on the grid.


3. For every sensor measurement calculate:


$$
posterior
=
previous\ posterior
\times
likelihood
$$


4. Normalize using trapezoidal integration:


$$
f(\theta)
=
\frac{f(\theta)}
{\int f(\theta)d\theta}
$$


5. Compute posterior mean and MAP.


6. Repeat for every new sensor reading.


"""))



def sensor_likelihood(theta,y,K,sigma):

    log_term = np.log(y)-np.log(theta*K)

    return (
        1/(y*sigma*np.sqrt(2*np.pi))
        *
        np.exp(-(log_term**2)/(2*sigma**2))
    )



theta_true = 0.68

K_nominal = 50.0

sigma = 0.15

n = 15



posterior = prior.copy()


posterior_history = [posterior.copy()]

bayes_estimates = [
    np.trapezoid(theta_grid*posterior,theta_grid)
]


map_estimates = [
    theta_grid[np.argmax(posterior)]
]


measurements=[]



for k in range(n):


    epsilon = np.random.normal(0,sigma)

    y = theta_true*K_nominal*np.exp(epsilon)

    measurements.append(y)



    likelihood = sensor_likelihood(
        theta_grid,
        y,
        K_nominal,
        sigma
    )


    posterior = posterior*likelihood


    posterior /= np.trapezoid(
        posterior,
        theta_grid
    )


    posterior_history.append(
        posterior.copy()
    )


    mean_value = np.trapezoid(
        theta_grid*posterior,
        theta_grid
    )


    map_value = theta_grid[
        np.argmax(posterior)
    ]


    bayes_estimates.append(mean_value)

    map_estimates.append(map_value)



display(Markdown(r"""

# Task 6: Posterior Evolution


"""))



milestones = [0,1,2,5,10,15]


fig = go.Figure()



for m in milestones:

    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_history[m],
            mode="lines",
            name=f"k={m}"
        )
    )



fig.update_layout(
    template="plotly_white",
    title="Posterior Density Evolution During Damage Detection",
    xaxis_title="Remaining Stiffness θ",
    yaxis_title="Posterior Density",
    height=600
)


fig.show()



steps=np.arange(0,n+1)



fig=go.Figure()


fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Posterior Mean"
    )
)



fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)



fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68"
)



fig.update_layout(
    template="plotly_white",
    title="Structural Stiffness Estimation Convergence",
    xaxis_title="Inspection Step",
    yaxis_title="Estimated θ",
    height=600
)


fig.show()



print("="*80)
print("SENSOR MEASUREMENTS")
print("="*80)

for i,value in enumerate(measurements):
    print(f"Measurement {i+1:2d}: {value:.3f} kN/mm")



print()

print("="*80)
print("FINAL ESTIMATES")
print("="*80)

print(f"True stiffness factor      : {theta_true:.3f}")
print(f"Posterior Mean Estimate   : {bayes_estimates[-1]:.3f}")
print(f"MAP Estimate              : {map_estimates[-1]:.3f}")



print()

print("="*80)
print("ENGINEERING ANALYSIS")
print("="*80)



print("""
Initially, the posterior distribution is concentrated near θ = 1
because the Beta(8,1.5) prior represents a healthy component.

As sensor measurements arrive, the likelihood contribution from
the degraded stiffness state shifts the posterior toward θ = 0.68.

After several measurements, the accumulated evidence becomes
stronger than the optimistic manufacturing prior.

The posterior density becomes narrower because uncertainty about
the remaining stiffness decreases.

A narrow posterior distribution indicates higher confidence in
structural condition assessment and allows engineers to define
more reliable safety thresholds.

The convergence of the Bayesian estimates demonstrates that
sequential sensor information can successfully detect gradual
structural degradation.
""")



# Bayesian Sequential Updating for Structural Health Monitoring (SHM)

## Task 1: Prior Belief Boundaries

The initial belief about the remaining stiffness efficiency is:

$$
\Theta \sim Beta(8,1.5)
$$


The probability density function is:

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1}
$$


The expected prior stiffness efficiency is:

$$
E[\Theta^{(0)}]
=
\frac{\alpha}{\alpha+\beta}
$$


For:

$$
\alpha=8,\beta=1.5
$$


$$
E[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
0.842
$$


This prior represents a healthy structure because the density is concentrated
near θ = 1, indicating that most manufactured components are expected to
operate close to their nominal stiffness.





# Task 2: Structural Likelihood Formulation


The sensor model is:

$$
y_k=\theta K_{nominal}e^{\epsilon_k}
$$


where:


$$
\epsilon_k\sim N(0,\sigma^2)
$$


Taking logarithms:


$$
ln(y_k)
=
ln(\theta K_{nominal})
+
\epsilon_k
$$


Therefore:


$$
ln(y_k)
\sim
N(ln(\theta K_{nominal}),\sigma^2)
$$


The likelihood of a single measurement is:


$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
exp
\left(
-\frac{
(ln(y_k)-ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right)
$$


The joint likelihood is:


$$
L(\mathbf y^{(k)}|\theta)
=
\prod_{i=1}^{k}
L(y_i|\theta)
$$






# Task 3: Non-Conjugate Bayesian Grid Update


The posterior recursion is:


$$
f(\theta|\mathbf y^{(k)})
\propto
L(y_k|\theta)
f(\theta|\mathbf y^{(k-1)})
$$


Unlike Beta-Binomial models, the Beta prior combined with a
log-normal likelihood does not produce another Beta distribution.

Therefore, a closed-form analytical posterior does not exist.


Numerical grid approximation is required to represent the posterior
distribution.





# Task 4: Running Point Estimates


The posterior mean is obtained using numerical integration:


$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{
\int_0^1
\theta f(\theta|\mathbf y^{(k)})d\theta
}
{
\int_0^1
f(\theta|\mathbf y^{(k)})d\theta
}
$$


The MAP estimate is:


$$
\hat{\theta}_{MAP}^{(k)}
=
\arg\max_{\theta}
f(\theta|\mathbf y^{(k)})
$$






# Task 5: Grid Approximation Algorithm


1. Construct a bounded grid:

$$
\theta \in [0.01,1]
$$


2. Initialize the Beta prior on the grid.


3. For every sensor measurement calculate:


$$
posterior
=
previous\ posterior
\times
likelihood
$$


4. Normalize using trapezoidal integration:


$$
f(\theta)
=
\frac{f(\theta)}
{\int f(\theta)d\theta}
$$


5. Compute posterior mean and MAP.


6. Repeat for every new sensor reading.






# Task 6: Posterior Evolution




SENSOR MEASUREMENTS
Measurement  1: 36.630 kN/mm
Measurement  2: 33.302 kN/mm
Measurement  3: 37.469 kN/mm
Measurement  4: 42.726 kN/mm
Measurement  5: 32.827 kN/mm
Measurement  6: 32.827 kN/mm
Measurement  7: 43.088 kN/mm
Measurement  8: 38.148 kN/mm
Measurement  9: 31.688 kN/mm
Measurement 10: 36.883 kN/mm
Measurement 11: 31.717 kN/mm
Measurement 12: 31.706 kN/mm
Measurement 13: 35.257 kN/mm
Measurement 14: 25.518 kN/mm
Measurement 15: 26.249 kN/mm

FINAL ESTIMATES
True stiffness factor      : 0.680
Posterior Mean Estimate   : 0.689
MAP Estimate              : 0.687

ENGINEERING ANALYSIS

Initially, the posterior distribution is concentrated near θ = 1
because the Beta(8,1.5) prior represents a healthy component.

As sensor measurements arrive, the likelihood contribution from
the degraded stiffness state shifts the posterior toward θ = 0.68.

After several measurements, the accumulated evidence becomes
stronger than the optimistic manufacturing prior.

The posterior density becomes 

Q. Gaussian Mixture Clustering as Conditional Updating

In [17]:
import numpy as np
import pandas as pd
import os

import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture

from IPython.display import display, Markdown

import kagglehub


class GMMFinancialSegmenter:

    def __init__(self, data, feature1, feature2, n_components=3):

        self.data = data

        self.features = [
            feature1,
            feature2
        ]

        self.X = data[self.features].dropna()


        self.scaler = StandardScaler()


        self.X_scaled = self.scaler.fit_transform(
            self.X
        )


        self.X_train, self.X_test = train_test_split(
            self.X_scaled,
            test_size=0.2,
            random_state=42
        )


        self.gmm = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=42
        )


        self.train_labels = None
        self.test_labels = None



    def fit(self):

        self.gmm.fit(
            self.X_train
        )


        self.train_labels = self.gmm.predict(
            self.X_train
        )


        self.test_labels = self.gmm.predict(
            self.X_test
        )


        print("="*70)
        print("GMM TRAINING RESULTS")
        print("="*70)


        print(
            "Converged:",
            self.gmm.converged_
        )


        print(
            "Iterations:",
            self.gmm.n_iter_
        )


        print(
            "Average Test Log Likelihood:",
            round(
                self.gmm.score(self.X_test),
                4
            )
        )



    def density_heatmap(self):

        X_original = self.scaler.inverse_transform(
            self.X_train
        )


        x = X_original[:,0]
        y = X_original[:,1]


        fig = go.Figure()


        fig.add_trace(
            go.Histogram2dContour(
                x=x,
                y=y,
                colorscale="Viridis",
                contours=dict(
                    coloring="fill"
                ),
                name="Density"
            )
        )


        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode="markers",
                marker=dict(
                    size=5
                ),
                name="Training Data"
            )
        )


        fig.update_layout(
            template="plotly_white",
            title="2D Density Heatmap of Training Data",
            xaxis_title=self.features[0],
            yaxis_title=self.features[1],
            height=600
        )


        fig.show()



    def responsibility_contour(self, dataset, title):


        x_min = dataset[:,0].min()-1
        x_max = dataset[:,0].max()+1

        y_min = dataset[:,1].min()-1
        y_max = dataset[:,1].max()+1


        xx,yy = np.meshgrid(
            np.linspace(x_min,x_max,200),
            np.linspace(y_min,y_max,200)
        )


        grid = np.column_stack(
            [
                xx.ravel(),
                yy.ravel()
            ]
        )


        probabilities = self.gmm.predict_proba(
            grid
        )


        clusters = np.argmax(
            probabilities,
            axis=1
        )


        zz = clusters.reshape(
            xx.shape
        )


        fig = go.Figure()


        fig.add_trace(
            go.Contour(
                x=np.linspace(x_min,x_max,200),
                y=np.linspace(y_min,y_max,200),
                z=zz,
                opacity=0.5,
                showscale=False,
                name="Cluster Regions"
            )
        )


        fig.add_trace(
            go.Scatter(
                x=dataset[:,0],
                y=dataset[:,1],
                mode="markers",
                marker=dict(
                    size=5
                ),
                name="Samples"
            )
        )


        fig.update_layout(
            template="plotly_white",
            title=title,
            xaxis_title=self.features[0]+" (scaled)",
            yaxis_title=self.features[1]+" (scaled)",
            height=600
        )


        fig.show()



    def training_assignment_plot(self):

        self.responsibility_contour(
            self.X_train,
            "Training Assignment Map"
        )



    def test_assignment_plot(self):

        self.responsibility_contour(
            self.X_test,
            "Test Assignment Map"
        )



display(
    Markdown(
r"""
# Part 10: Computational Simulation and Out-of-Sample Validation

The Gaussian Mixture Model is trained using the Expectation-Maximization algorithm.

The EM algorithm consists of:

### E-Step

Computing posterior probabilities:

$$
\gamma_{ik}=P(C_i=k|X_i=x_i)
$$


### M-Step

Updating Gaussian parameters:

$$
\mu_k=
\frac{1}{N_k}
\sum_i \gamma_{ik}x_i
$$


The final soft assignment expectation is:

$$
E[Z_i|X_i=x_i]
$$

"""
    )
)



path = kagglehub.dataset_download(
    "arjunbhasin2013/ccdata"
)


print("Dataset Location:")
print(path)



csv_path = None


for root,dirs,files in os.walk(path):

    for file in files:

        if file.lower()=="cc general.csv":

            csv_path=os.path.join(
                root,
                file
            )


if csv_path is None:
    raise FileNotFoundError(
        "CSV dataset not found"
    )


print("CSV File:")
print(csv_path)



data = pd.read_csv(
    csv_path
)


print("="*70)
print("DATASET INFORMATION")
print("="*70)


display(data.head())


print("Dataset Shape:")
print(data.shape)



display(
    Markdown(
r"""
## Feature Selection

Selected variables:

- PURCHASES
- CREDIT_LIMIT

These variables represent customer spending behaviour and available
credit capacity.

The variables are standardized before applying GMM.
"""
    )
)



model = GMMFinancialSegmenter(
    data,
    "PURCHASES",
    "CREDIT_LIMIT",
    n_components=3
)



model.fit()



display(
    Markdown(
r"""
## Visualization 1: Density Heatmap

The density map shows regions of high concentration in the financial
feature space.
"""
    )
)


model.density_heatmap()



display(
    Markdown(
r"""
## Visualization 2: Training Assignment Map

Each region represents the cluster with the highest posterior
probability.

$$
\hat C_i=argmax_k(\gamma_{ik})
$$
"""
    )
)


model.training_assignment_plot()



display(
    Markdown(
r"""
## Visualization 3: Test Assignment Map

The unseen test samples are classified using the learned Gaussian
mixture probability structure.
"""
    )
)


model.test_assignment_plot()



print("="*80)

print(
"""
FINAL INTERPRETATION

The Gaussian Mixture Model performs probabilistic clustering.

Each observation receives a responsibility value representing its
probability of belonging to each customer segment.

The Expectation-Maximization algorithm estimates:

- Cluster probabilities
- Gaussian means
- Covariance matrices

The density contours show the hidden financial segments.

Similar training and testing assignment patterns indicate that the
model generalizes well to unseen customer data.
"""
)


# Part 10: Computational Simulation and Out-of-Sample Validation

The Gaussian Mixture Model is trained using the Expectation-Maximization algorithm.

The EM algorithm consists of:

### E-Step

Computing posterior probabilities:

$$
\gamma_{ik}=P(C_i=k|X_i=x_i)
$$


### M-Step

Updating Gaussian parameters:

$$
\mu_k=
\frac{1}{N_k}
\sum_i \gamma_{ik}x_i
$$


The final soft assignment expectation is:

$$
E[Z_i|X_i=x_i]
$$



Using Colab cache for faster access to the 'ccdata' dataset.
Dataset Location:
/kaggle/input/ccdata
CSV File:
/kaggle/input/ccdata/CC GENERAL.csv
DATASET INFORMATION


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


Dataset Shape:
(8950, 18)



## Feature Selection

Selected variables:

- PURCHASES
- CREDIT_LIMIT

These variables represent customer spending behaviour and available
credit capacity.

The variables are standardized before applying GMM.


GMM TRAINING RESULTS
Converged: True
Iterations: 19
Average Test Log Likelihood: -1.6465



## Visualization 1: Density Heatmap

The density map shows regions of high concentration in the financial
feature space.



## Visualization 2: Training Assignment Map

Each region represents the cluster with the highest posterior
probability.

$$
\hat C_i=argmax_k(\gamma_{ik})
$$



## Visualization 3: Test Assignment Map

The unseen test samples are classified using the learned Gaussian
mixture probability structure.



FINAL INTERPRETATION

The Gaussian Mixture Model performs probabilistic clustering.

Each observation receives a responsibility value representing its
probability of belonging to each customer segment.

The Expectation-Maximization algorithm estimates:

- Cluster probabilities
- Gaussian means
- Covariance matrices

The density contours show the hidden financial segments.

Similar training and testing assignment patterns indicate that the
model generalizes well to unseen customer data.

